In [1]:
%pip install -q "pgmpy>=1.0" numpy pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 8.3 MB/s eta 0:00:00


In [2]:
import itertools
import numpy as np
import pandas as pd

from IPython.display import display
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

reference_model = DiscreteBayesianNetwork([
    ("Cloudy", "Rain"),
    ("Cloudy", "Sprinkler"),
    ("Rain", "WetGrass"),
    ("Sprinkler", "WetGrass"),
])

cpd_cloudy = TabularCPD(
    variable="Cloudy",
    variable_card=2,
    values=[[0.5], [0.5]],
)

cpd_rain = TabularCPD(
    variable="Rain",
    variable_card=2,
    values=[
        [0.8, 0.2],
        [0.2, 0.8],
    ],
    evidence=["Cloudy"],
    evidence_card=[2],
)

cpd_sprinkler = TabularCPD(
    variable="Sprinkler",
    variable_card=2,
    values=[
        [0.5, 0.9],
        [0.5, 0.1],
    ],
    evidence=["Cloudy"],
    evidence_card=[2],
)

cpd_wetgrass = TabularCPD(
    variable="WetGrass",
    variable_card=2,
    values=[
        [0.99, 0.10, 0.10, 0.01],
        [0.01, 0.90, 0.90, 0.99],
    ],
    evidence=["Rain", "Sprinkler"],
    evidence_card=[2, 2],
)

reference_model.add_cpds(
    cpd_cloudy,
    cpd_rain,
    cpd_sprinkler,
    cpd_wetgrass,
)

assert reference_model.check_model()

print("Model valid:", reference_model.check_model())

for cpd in reference_model.get_cpds():
    print(cpd)
    print()

/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Model valid: True
+-----------+-----+
| Cloudy(0) | 0.5 |
+-----------+-----+
| Cloudy(1) | 0.5 |
+-----------+-----+

+---------+-----------+-----------+
| Cloudy  | Cloudy(0) | Cloudy(1) |
+---------+-----------+-----------+
| Rain(0) | 0.8       | 0.2       |
+---------+-----------+-----------+
| Rain(1) | 0.2       | 0.8       |
+---------+-----------+-----------+

+--------------+-----------+-----------+
| Cloudy       | Cloudy(0) | Cloudy(1) |
+--------------+-----------+-----------+
| Sprinkler(0) | 0.5       | 0.9       |
+--------------+-----------+-----------+
| Sprinkler(1) | 0.5       | 0.1       |
+--------------+-----------+-----------+

+-------------+--------------+--------------+--------------+--------------+
| Rain        | Rain(0)      | Rain(0)      | Rain(1)      | Rain(1)      |
+-------------+--------------+--------------+--------------+--------------+
| Sprinkler   | Sprinkler(0) | Sprinkler(1) | Sprinkler(0) | Sprinkler(1) |
+-------------+--------------+------

## HW Exercise: Change the Query

All variables use states 0 = False and 1 = True.

### Query 1: P(Sprinkler = 1 | WetGrass = 1)

Query variable: Sprinkler.
Evidence: WetGrass = 1.

Prediction: The probability should increase from its prior of 0.3,
because the sprinkler is a possible cause of wet grass.

Code inspection: The query must contain Sprinkler as the query
variable and {"WetGrass": 1} as the evidence.

### Query 2: P(Cloudy = 1 | WetGrass = 1)

Query variable: Cloudy.
Evidence: WetGrass = 1.

Prediction: Under the given CPTs, a modest increase from the prior
of 0.5 is expected. Cloudiness increases the probability of rain
but decreases the probability that the sprinkler is on.

Code inspection: Cloudy must be the query variable, and WetGrass
must be supplied as evidence with state 1.

### Query 3: P(Rain = 1 | WetGrass = 1, Sprinkler = 0)

Query variable: Rain.
Evidence: WetGrass = 1 and Sprinkler = 0.

Prediction: Rain should become much more likely because the grass
is wet while the sprinkler explanation has been ruled out.

Code inspection: Both evidence variables must appear, with
Sprinkler set to 0. The network must remain unchanged.

In [3]:
query1_result = VariableElimination(reference_model).query(
    variables=["Sprinkler"],
    evidence={"WetGrass": 1},
    show_progress=False,
)

query2_result = VariableElimination(reference_model).query(
    variables=["Cloudy"],
    evidence={"WetGrass": 1},
    show_progress=False,
)

query3_result = VariableElimination(reference_model).query(
    variables=["Rain"],
    evidence={"WetGrass": 1, "Sprinkler": 0},
    show_progress=False,
)

print(query1_result)
print(
    "P(Sprinkler=1 | WetGrass=1) =",
    query1_result.get_value(Sprinkler=1),
)

print(query2_result)
print(
    "P(Cloudy=1 | WetGrass=1) =",
    query2_result.get_value(Cloudy=1),
)

print(query3_result)
print(
    "P(Rain=1 | WetGrass=1, Sprinkler=0) =",
    query3_result.get_value(Rain=1),
)

+--------------+------------------+
| Sprinkler    |   phi(Sprinkler) |
+==============+==================+
| Sprinkler(0) |           0.5722 |
+--------------+------------------+
| Sprinkler(1) |           0.4278 |
+--------------+------------------+
P(Sprinkler=1 | WetGrass=1) = 0.4278461538461539
+-----------+---------------+
| Cloudy    |   phi(Cloudy) |
+===========+===============+
| Cloudy(0) |        0.4254 |
+-----------+---------------+
| Cloudy(1) |        0.5746 |
+-----------+---------------+
P(Cloudy=1 | WetGrass=1) = 0.5746153846153845
+---------+-------------+
| Rain    |   phi(Rain) |
+=========+=============+
| Rain(0) |      0.0078 |
+---------+-------------+
| Rain(1) |      0.9922 |
+---------+-------------+
P(Rain=1 | WetGrass=1, Sprinkler=0) = 0.9922022048937886


In [4]:
def joint_probability(c, r, s, w):
    p_r = {0: 0.2, 1: 0.8}[c]
    p_s = {0: 0.5, 1: 0.1}[c]
    p_w = {
        (0, 0): 0.01,
        (0, 1): 0.90,
        (1, 0): 0.90,
        (1, 1): 0.99,
    }[(r, s)]

    return (
        0.5
        * (p_r if r == 1 else 1 - p_r)
        * (p_s if s == 1 else 1 - p_s)
        * (p_w if w == 1 else 1 - p_w)
    )


def enumerate_posterior(variable, evidence):
    weights = np.zeros(2)

    for c, r, s, w in itertools.product([0, 1], repeat=4):
        assignment = {
            "Cloudy": c,
            "Rain": r,
            "Sprinkler": s,
            "WetGrass": w,
        }

        if all(
            assignment[name] == state
            for name, state in evidence.items()
        ):
            weights[assignment[variable]] += joint_probability(c, r, s, w)

    return weights / weights.sum()


trusted_infer = VariableElimination(reference_model)

queries = [
    ("Sprinkler", {"WetGrass": 1}, query1_result),
    ("Cloudy", {"WetGrass": 1}, query2_result),
    ("Rain", {"WetGrass": 1, "Sprinkler": 0}, query3_result),
]

rows = []

for variable, evidence, candidate in queries:
    trusted = trusted_infer.query(
        variables=[variable],
        evidence=evidence,
        show_progress=False,
    )

    candidate_values = np.array([
        candidate.get_value(**{variable: state})
        for state in [0, 1]
    ])

    trusted_values = np.array([
        trusted.get_value(**{variable: state})
        for state in [0, 1]
    ])

    enumerated_values = enumerate_posterior(variable, evidence)

    assert np.isclose(candidate_values.sum(), 1)
    assert np.allclose(candidate_values, trusted_values)
    assert np.allclose(candidate_values, enumerated_values)

    rows.append({
        "Query variable": variable,
        "Evidence": str(evidence),
        "Candidate P(1)": candidate_values[1],
        "Trusted P(1)": trusted_values[1],
        "Enumeration P(1)": enumerated_values[1],
        "Result": "PASS",
    })

display(pd.DataFrame(rows))
print("All three queries passed validation.")

,Query variable,Evidence,Candidate P(1),Trusted P(1),Enumeration P(1),Result
0,Sprinkler,{'WetGrass': 1},0.427846,0.427846,0.427846,PASS
1,Cloudy,{'WetGrass': 1},0.574615,0.574615,0.574615,PASS
2,Rain,"{'WetGrass': 1, 'Sprinkler': 0}",0.992202,0.992202,0.992202,PASS


All three queries passed validation.


## Interpretation of Results

1. P(Sprinkler = 1 | WetGrass = 1) ≈ 0.427846.
   This exceeds the prior probability of 0.3, matching the prediction.

2. P(Cloudy = 1 | WetGrass = 1) ≈ 0.574615.
   This is a modest increase from the prior probability of 0.5.

3. P(Rain = 1 | WetGrass = 1, Sprinkler = 0) ≈ 0.992202.
   Wet grass with the sprinkler off strongly supports rain.
   The probability is below 1 because wet grass is still possible
   when both rain and the sprinkler are absent.

The candidate code was generated using ChatGPT. The notebook checks
its results against trusted VariableElimination calls and independent
enumeration of the joint distribution. Agreement between two pgmpy
calls alone is not an independent mathematical validation.